# Notebook 12: ArchiMate Preparation Current-Target (Actual/Target)

Preparation, loading data, consolidating, exporting:
- Load all relevant baseline and profile extension outputs (Target baseline, extended 1.1, extended 2.1)
- Generate a uniform long format for each KldB-5 code (normalized skill ID + source flag)
- Enable case selection (search for KldB code by job title/description)
- Generate ArchiMate input tables (elements + skill lists + prioritization)
- Export as CSV/Excel to enable quick ArchiMate modeling in Archi

**Overall result:** Creation of the complete consolidated results table (basic profiles + extensions) in step 6.1
- https://www.archimatetool.com/, February 10, 2026

## 1. Setup & Paths

In [1]:
# Setup + Paths
from pathlib import Path
import os
import json
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Project Root
PROJECT_ROOT = Path(".").resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
DATA_INTERIM = DATA / "interim"
DATA_PROCESSED = DATA / "processed"
DATA_PROCESSED_EXTERNAL = DATA / "processed_external"

# New: ArchiMate Outputs
DATA_PROCESSED_ARCHI = DATA / "processed_archimate"
DATA_PROCESSED_ARCHI.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PROCESSED_ARCHI:", DATA_PROCESSED_ARCHI)

PROJECT_ROOT: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung
DATA_PROCESSED_ARCHI: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate


Input paths:

In [2]:
# Baseline Target (SOLL)
KDB_SOLL_LONG_PATH = DATA_PROCESSED / "kldb_skills_soll_long.parquet"
KDB_SOLL_AGG_PATH  = DATA_PROCESSED / "kldb_skills_soll_agg.parquet"

# Extended Target (SOLL) aus 1.1
SOLL_LONG_EXT_11_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skills_soll_long_extended_rulebased.parquet"
SOLL_AGG_EXT_11_PATH  = DATA_PROCESSED_EXTERNAL / "kldb_skills_soll_agg_extended_rulebased.parquet"

# Extended Target (SOLL) aus 2.1
METHOD_21_DIR = DATA_PROCESSED_EXTERNAL / "method_2_1"
SOLL_LONG_EXT_21_PATH = METHOD_21_DIR / "kldb_skills_soll_long_extended_2_1.parquet"
SOLL_AGG_EXT_21_PATH  = METHOD_21_DIR / "kldb_skills_soll_agg_extended_2_1.parquet"

# Skills Concepts (prefer filtered)
CONCEPTS_CANDIDATES = [
    DATA_PROCESSED_EXTERNAL / "skills_concepts_filtered.parquet",
    DATA_PROCESSED_EXTERNAL / "skills_concepts.parquet",]
CONCEPTS_PATH = next((p for p in CONCEPTS_CANDIDATES if p.exists()), None)

# Candidate Pool Jobtitel -> KldB
CANDIDATES_PATH = DATA_INTERIM / "kldb_job_title_candidates_de_en.parquet"

# Job title (DE) from NB04
JOB_TITLES_LONG_PATH = DATA_PROCESSED / "kldb_job_titles_long.parquet"
JOB_TITLES_AGG_PATH  = DATA_PROCESSED / "kldb_job_titles_agg.parquet"

# ESCO/KldB Mapping
KLDB_ESCO_MAP_PATH = DATA_INTERIM / "kldb_esco_mapping.parquet"
ESCO_OCC_PATH = DATA_INTERIM / "esco_occupations.parquet"
ESCO_SKILLS_PATH = DATA_INTERIM / "esco_skills.parquet"
paths_required = [KDB_SOLL_LONG_PATH, SOLL_LONG_EXT_11_PATH, SOLL_LONG_EXT_21_PATH, CANDIDATES_PATH,]

print("OK alle Pflichtpfade da:")
print("CONCEPTS_PATH:", CONCEPTS_PATH)

OK alle Pflichtpfade da:
CONCEPTS_PATH: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_concepts_filtered.parquet


## 2. Normalization (KldB-Code, Skill-ID, Text)

In [3]:
def norm_kldb_code(x) -> str | None:
    if pd.isna(x):
        return None
    s = str(x).strip()
    m = re.search(r"(\d{5})", s)
    if not m:
        return None
    return m.group(1)

def norm_whitespace(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()

def to_skill_id(x) -> str | None: # Standardizes skill IDs: Leave the prefix (ESCO:/BA:/LINKEDIN:/RESUME:) as is; URL/URI -> ESCO:<uri>; otherwise string trimmed
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None
    up = s.upper()
    if up.startswith(("ESCO:", "BA:", "LINKEDIN:", "RESUME:")):
        return s
    if s.startswith("http://") or s.startswith("https://"):
        return "ESCO:" + s
    return s

def pick_first_existing_col(df: pd.DataFrame, candidates: list[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Keine der Spalten gefunden: {candidates}")

## 3. Load Core Target Tables

- Baseline Target (Notebook 04)
- Extended Target 1.1 (rule-based)
- Extended Target 2.1 (text mining)

Then standardize:
- kldb_5_code (string, 5 characters)
- skill_id (always with prefix)
- source flags: baseline/ext_1_1/ext_2_1

In [4]:
df_soll_base = pd.read_parquet(KDB_SOLL_LONG_PATH)
df_ext_11 = pd.read_parquet(SOLL_LONG_EXT_11_PATH)
df_ext_21 = pd.read_parquet(SOLL_LONG_EXT_21_PATH)

print("BASE:", df_soll_base.shape, "cols:", len(df_soll_base.columns))
print("EXT 1.1:", df_ext_11.shape, "cols:", len(df_ext_11.columns))
print("EXT 2.1:", df_ext_21.shape, "cols:", len(df_ext_21.columns))

display(df_soll_base.head(3))
display(df_ext_11.head(3))
display(df_ext_21.head(3))

BASE: (809412, 20) cols: 20
EXT 1.1: (959410, 27) cols: 27
EXT 2.1: (831755, 25) cols: 25


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count,occupation_uri,occupation_code,isco_group,occupation_title_en,occupation_description_en,skill_uri,skill_title_en,skill_description_en,reuse_level,skill_type,relation_type,kldb_job_titles_de
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0b34dd9f-bcb2...,apply health and safety when picking,Take the necessary health and safety precautio...,occupation-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf..."
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0e355580-bd4a...,work in outdoor conditions,Can cope with the different climate conditions...,cross-sector,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf..."
2,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/2e4e8114-e579...,harvest crop,"Mow, pick or cut agricultural crop products ma...",sector-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf..."


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count,occupation_uri,occupation_code,isco_group,occupation_title_en,occupation_description_en,skill_uri,skill_title_en,skill_description_en,reuse_level,skill_type,relation_type,kldb_job_titles_de,skill_id,is_extension,extension_method,ext_doc_freq,ext_skill_count,ext_example_label,ext_example_source
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0b34dd9f-bcb2...,apply health and safety when picking,Take the necessary health and safety precautio...,occupation-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",ESCO:http://data.europa.eu/esco/skill/0b34dd9f...,False,None,NaN,NaN,None,None
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0e355580-bd4a...,work in outdoor conditions,Can cope with the different climate conditions...,cross-sector,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",ESCO:http://data.europa.eu/esco/skill/0e355580...,False,None,NaN,NaN,None,None
2,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/2e4e8114-e579...,harvest crop,"Mow, pick or cut agricultural crop products ma...",sector-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",ESCO:http://data.europa.eu/esco/skill/2e4e8114...,False,None,NaN,NaN,None,None


,kldb_5_code,kldb_title_de,kldb_title_en,isco08_4,isco_title_de,isco_title_en,unambiguous_flag,focus_and_alt_count,occupation_uri,occupation_code,isco_group,occupation_title_en,occupation_description_en,skill_uri,skill_title_en,skill_description_en,reuse_level,skill_type,relation_type,kldb_job_titles_de,is_extension,extension_method,doc_freq,skill_count,example_label
0,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0b34dd9f-bcb2...,apply health and safety when picking,Take the necessary health and safety precautio...,occupation-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",False,None,NaN,NaN,None
1,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/0e355580-bd4a...,work in outdoor conditions,Can cope with the different climate conditions...,cross-sector,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",False,None,NaN,NaN,None
2,11101,Berufe in der Landwirtschaft (ohne Spezialisie...,Occupations in farming (without specialisation...,9211,Hilfsarbeiter im Ackerbau,Crop farm labourers,0,1,http://data.europa.eu/esco/occupation/373db468...,9211.1,9211,fruit and vegetable picker,Fruit and vegetable pickers select and harvest...,http://data.europa.eu/esco/skill/2e4e8114-e579...,harvest crop,"Mow, pick or cut agricultural crop products ma...",sector-specific,skill/competence,essential,"Ackerarbeiter/in, Ackergehilf(e/in), Erntehelf...",False,None,NaN,NaN,None


## 4. Column Standardization: Identify Skill Columns + Convert to `skill_id`

In Step 4, the skill tables from the three sources (Baseline, Extension 1.1, Extension 2.1) are converted into a uniform schema:
- `kldb_5_code` is normalized to 5 digits
- `skill_id` is robustly standardized (ESCO/BA/LINKEDIN/RESUME; URLs are marked as ESCO IDs)
- Source flags are set (`src_baseline`, `src_ext_1_1`, `src_ext_2_1`)
- A preliminary check is then performed to verify whether the normalization was successful (e.g., >95% non-zero)

In [5]:
# Baseline
base_skill_col = pick_first_existing_col(df_soll_base, ["skill_id", "skill_uri", "skill"])
df_soll_base = df_soll_base.copy()
df_soll_base["kldb_5_code"] = df_soll_base["kldb_5_code"].apply(norm_kldb_code)
df_soll_base["skill_id"] = df_soll_base[base_skill_col].apply(to_skill_id)

# Baseline Flag
df_soll_base["src_baseline"] = True
df_soll_base["src_ext_1_1"] = False
df_soll_base["src_ext_2_1"] = False
df_soll_base["src_name"] = "baseline_esco"

# EXT 1.1
ext11_skill_col = pick_first_existing_col(df_ext_11, ["skill_id", "skill_uri", "skill"])
df_ext_11 = df_ext_11.copy()
df_ext_11["kldb_5_code"] = df_ext_11["kldb_5_code"].apply(norm_kldb_code)
df_ext_11["skill_id"] = df_ext_11[ext11_skill_col].apply(to_skill_id)

# use is_extension, otherwise as extended file
if "is_extension" in df_ext_11.columns:
    df_ext_11["src_baseline"] = ~df_ext_11["is_extension"].fillna(False)
    df_ext_11["src_ext_1_1"] = df_ext_11["is_extension"].fillna(False)
else:
    df_ext_11["src_baseline"] = False
    df_ext_11["src_ext_1_1"] = True

df_ext_11["src_ext_2_1"] = False
df_ext_11["src_name"] = "extended_rule_based_1_1"

# EXT 2.1
ext21_skill_col = pick_first_existing_col(df_ext_21, ["skill_id", "skill_uri", "skill"])
df_ext_21 = df_ext_21.copy()
df_ext_21["kldb_5_code"] = df_ext_21["kldb_5_code"].apply(norm_kldb_code)
df_ext_21["skill_id"] = df_ext_21[ext21_skill_col].apply(to_skill_id)

if "is_extension" in df_ext_21.columns: # same as 1.1
    df_ext_21["src_baseline"] = ~df_ext_21["is_extension"].fillna(False)
    df_ext_21["src_ext_2_1"] = df_ext_21["is_extension"].fillna(False)
else:
    df_ext_21["src_baseline"] = False
    df_ext_21["src_ext_2_1"] = True

df_ext_21["src_ext_1_1"] = False
df_ext_21["src_name"] = "extended_text_mining_2_1"

# Check
assert df_soll_base["kldb_5_code"].notna().mean() > 0.95
assert df_soll_base["skill_id"].notna().mean() > 0.95

print("OK: Standardisierung done.")
print("BASE unique kldb:", df_soll_base["kldb_5_code"].nunique(), "; unique skill:", df_soll_base["skill_id"].nunique())
print("EXT11 unique kldb:", df_ext_11["kldb_5_code"].nunique(), "; unique skill:", df_ext_11["skill_id"].nunique())
print("EXT21 unique kldb:", df_ext_21["kldb_5_code"].nunique(), "; unique skill:", df_ext_21["skill_id"].nunique())

OK: Standardisierung done.
BASE unique kldb: 1300 ; unique skill: 13471
EXT11 unique kldb: 1300 ; unique skill: 16493
EXT21 unique kldb: 1300 ; unique skill: 20965


Output:
- `BASE unique kldb`/`EXT.. unique kldb`: The number of distinct KldB codes for which skills are available in the respective table.
- `unique kldb` is 1,300 in all cases here, since they always include the standard skills as well, meaning all KLDB codes are represented. The number of codes that were actually expanded in the two expansion methods is 894 for Method 1.1 (Notebook 10b) and 352 for Method 2.1 (Notebook 11).
- `unique skill`: The total number of distinct skills found in this source (across all KLDB codes). Although Method 2.1 expands fewer codes, it has a more precise and therefore broader skill base.

## 5. Skill Labels

### 5.1 Load Concepts + Join

Load skill concepts: join skill_id -> display_label (+ source/language). For later exports to ArchiMate.

In [6]:
df_concepts = pd.read_parquet(CONCEPTS_PATH)
print("CONCEPTS:", df_concepts.shape)
display(df_concepts.head(3))

concept_skill_col = pick_first_existing_col(df_concepts, ["skill_id", "source_id", "skill_uri"]) # Spalten
concept_label_col = pick_first_existing_col(df_concepts, ["display_label", "label", "example_label", "display_label_fallback"])

df_concepts = df_concepts.copy()
df_concepts["skill_id"] = df_concepts[concept_skill_col].apply(to_skill_id)
df_concepts["skill_label"] = df_concepts[concept_label_col].apply(norm_whitespace)

keep_cols = ["skill_id", "skill_label"]
for c in ["source_system", "display_source", "lang", "display_lang"]:
    if c in df_concepts.columns:
        keep_cols.append(c)

df_concepts_small = (
    df_concepts[keep_cols]
    .dropna(subset=["skill_id"])
    .drop_duplicates(subset=["skill_id"])
    .reset_index(drop=True)
)

print("CONCEPTS small:", df_concepts_small.shape)
display(df_concepts_small.sample(5, random_state=42))

CONCEPTS: (58303, 4)


,skill_id,display_label,source_system,lang
0,ESCO:http://data.europa.eu/esco/skill/51586df8...,R,ESCO,en
1,ESCO:http://data.europa.eu/esco/skill/4c016b68...,C#,ESCO,en
2,ESCO:http://data.europa.eu/esco/skill/58d7a289...,APL,ESCO,en


CONCEPTS small: (58303, 4)


,skill_id,skill_label,source_system,lang
35008,LINKEDIN:rec to rec,Rec to Rec,LINKEDIN,en
20090,BA:K 131202-024,Wohngebäudeversicherung,BA,de
52794,LINKEDIN:personalized medicine,Personalized Medicine,LINKEDIN,en
1414,ESCO:http://data.europa.eu/esco/skill/637ca2e2...,proofread text,ESCO,en
32104,LINKEDIN:safeboot,Safeboot,LINKEDIN,en


In [7]:
# Join
def enrich_with_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.merge(df_concepts_small, on="skill_id", how="left")
    out["skill_label"] = out["skill_label"].fillna(out["skill_id"]) # falls kein Label skill_id anzeigen
    return out

df_soll_base_l = enrich_with_labels(df_soll_base)
df_ext_11_l = enrich_with_labels(df_ext_11)
df_ext_21_l = enrich_with_labels(df_ext_21)

print("enriched OK")

enriched OK


### 5.2 Language: German Skill Labels (ESCO)

For ArchiMate modeling and subsequent skill matching, all skills are additionally annotated with German labels. Since both the current and target models contain German job titles and skills:
- The skill ID remains the key
- English labels are also retained
- A new column, `skill_label_de`, is added
- Basis: official ESCO-DE skill datasets (located at data/raw/esco/esco_de; source: https://esco.ec.europa.eu/en/use-esco/download; v.1.2.0)

In [8]:
# load ESCO-DE Skills 
ESCO_DE_SKILLS_PATH = DATA / "raw" / "esco" / "esco_de" / "skills_de.csv"

df_skills_de = pd.read_csv(ESCO_DE_SKILLS_PATH)
print(df_skills_de.shape)
df_skills_de.head()

(13939, 13)


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,Musikpersonal verwalten,NaN,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Zuweisen und Verwalten der Aufgaben des Person...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,Strafvollzugsverfahren beaufsichtigen,NaN,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Überwachen des Betriebs einer Justizvollzugsan...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,nicht unterdrückende Praktiken anwenden,NaN,NaN,released,2023-11-28T10:45:53.54Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Ermitteln von Repressionen in Gesellschaften, ..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,Einhaltung von Vorschriften von Eisenbahnfahrz...,NaN,NaN,released,2023-11-30T16:29:18.273Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Kontrollieren von Fahrzeugen, Komponenten und ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,verfügbare Dienste ermitteln,NaN,NaN,released,2023-11-28T10:38:49.206Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Ermitteln der verschiedenen verfügbaren Dienst...


Mapping Table:

In [9]:
# Mapping Skill-ID -> German Label
df_skill_de_map = (df_skills_de.assign(skill_id=df_skills_de["conceptUri"].apply(to_skill_id))
              .rename(columns={"preferredLabel": "skill_label_de"})[["skill_id", "skill_label_de"]]
              .dropna(subset=["skill_id"])
              .drop_duplicates(subset=["skill_id"])
)

print("DE map:", df_skill_de_map.shape)
display(df_skill_de_map.head(5))

DE map: (13939, 2)


,skill_id,skill_label_de
0,ESCO:http://data.europa.eu/esco/skill/0005c151...,Musikpersonal verwalten
1,ESCO:http://data.europa.eu/esco/skill/00064735...,Strafvollzugsverfahren beaufsichtigen
2,ESCO:http://data.europa.eu/esco/skill/000709ed...,nicht unterdrückende Praktiken anwenden
3,ESCO:http://data.europa.eu/esco/skill/0007bdc2...,Einhaltung von Vorschriften von Eisenbahnfahrz...
4,ESCO:http://data.europa.eu/esco/skill/00090cc1...,verfügbare Dienste ermitteln


Function: Merge German labels into all skill tables:

In [10]:
def add_german_labels(df: pd.DataFrame, fallback_col: str = "skill_label") -> pd.DataFrame:
    out = df.copy()

    if fallback_col not in out.columns: # fallback_col
        for c in ["skill_label", "display_label", "label", "skill", "skill_id"]: # A valid field as a fallback to prevent a KeyError
            if c in out.columns:
                fallback_col = c
                break

    out = out.merge(df_skill_de_map, on="skill_id", how="left")
    out["skill_label_de"] = out["skill_label_de"].fillna(out[fallback_col]) # If no German label is found, keep the original label
    return out

In [11]:
# apply to DF's from 5.1
df_soll_base_l = add_german_labels(enrich_with_labels(df_soll_base))
df_ext_11_l    = add_german_labels(enrich_with_labels(df_ext_11))
df_ext_21_l    = add_german_labels(enrich_with_labels(df_ext_21))

# checks
assert "skill_label_de" in df_soll_base_l.columns
assert "skill_label_de" in df_ext_11_l.columns
assert "skill_label_de" in df_ext_21_l.columns

## 6. Master Skill Long for ArchiMate

Consolidated table that aggregates by (kldb_5_code, skill_id):
- Baseline
- Extension 1.1
- Extension 2.1
- Meta/Weights (Relation type/doc_freq, etc.)

In [12]:
# Fields that are important for ArchiMate
base_cols = ["kldb_5_code", "skill_id", "skill_label", "skill_label_de", "src_baseline", "src_ext_1_1", "src_ext_2_1", "src_name"]
# Baseline relation_type
for c in ["relation_type", "skill_type", "reuse_level"]:
    if c in df_soll_base_l.columns and c not in base_cols:
        base_cols.append(c)

ext_cols = ["kldb_5_code", "skill_id", "skill_label", "skill_label_de", "src_baseline", "src_ext_1_1", "src_ext_2_1", "src_name"]
# The extension has ext_doc_freq/ext_skill_count
for c in ["extension_method", "ext_doc_freq", "ext_skill_count", "doc_freq", "skill_count", "map_score", "map_reason"]:
    if c in df_ext_11_l.columns and c not in ext_cols:
        ext_cols.append(c)

df_base_min = df_soll_base_l[[c for c in base_cols if c in df_soll_base_l.columns]].copy()
df_11_min   = df_ext_11_l[[c for c in ext_cols if c in df_ext_11_l.columns]].copy()
df_21_min   = df_ext_21_l[[c for c in ext_cols if c in df_ext_21_l.columns]].copy()

# raw master long
df_master_long = pd.concat([df_base_min, df_11_min, df_21_min], ignore_index=True)

# Standardize Flags
for flag in ["src_baseline", "src_ext_1_1", "src_ext_2_1"]:
    if flag in df_master_long.columns:
        df_master_long[flag] = df_master_long[flag].fillna(False).astype(bool)

def first_non_null(s):
    s = s.dropna()
    return s.iloc[0] if len(s) else None

# Dedupe, Aggregate flags and keep the best label
agg_dict = {
    "skill_label": first_non_null,
    "skill_label_de": first_non_null,
    "src_baseline": "max",
    "src_ext_1_1": "max",
    "src_ext_2_1": "max",
}

# Aggregate only columns that actually exist
agg_dict = {k: v for k, v in agg_dict.items() if k in df_master_long.columns}

# Additional Fields
for c in ["relation_type", "skill_type", "reuse_level", "extension_method"]:
    if c in df_master_long.columns:
        agg_dict[c] = "first"
for c in ["ext_doc_freq", "ext_skill_count", "doc_freq", "skill_count"]:
    if c in df_master_long.columns:
        agg_dict[c] = "max"

df_master = (
    df_master_long
    .dropna(subset=["kldb_5_code", "skill_id"])
    .groupby(["kldb_5_code", "skill_id"], as_index=False)
    .agg(agg_dict)
)

print("MASTER:", df_master.shape)
display(df_master.sample(10, random_state=42))

MASTER: (553120, 13)


,kldb_5_code,skill_id,skill_label,skill_label_de,src_baseline,src_ext_1_1,src_ext_2_1,relation_type,skill_type,reuse_level,extension_method,ext_doc_freq,ext_skill_count
543881,94512,ESCO:http://data.europa.eu/esco/skill/81079e09...,coordinate pre-show checks,Kontrollen vor der Veranstaltung koordinieren,True,False,False,essential,skill/competence,sector-specific,None,NaN,NaN
135724,27302,ESCO:http://data.europa.eu/esco/skill/6dce7757...,be at ease in unsafe environments,in unsicherer Umgebung gelassen bleiben,True,False,False,optional,skill/competence,cross-sector,None,NaN,NaN
502404,91354,ESCO:http://data.europa.eu/esco/skill/410540a4...,player logic,Spielerlogik,True,False,False,essential,knowledge,occupation-specific,None,NaN,NaN
209570,33102,BA:K 030300-025,Management,Management,False,True,False,None,None,None,rule_based_1_1,5.0,5.0
157040,28232,BA:K 070401-099,SAP-Basis-System,SAP-Basis-System,False,True,False,None,None,None,rule_based_1_1,39.0,39.0
235199,41104,ESCO:http://data.europa.eu/esco/skill/0d7826c4...,develop gambling games,Glücksspiele entwickeln,True,False,False,essential,skill/competence,occupation-specific,None,NaN,NaN
402405,72123,ESCO:http://data.europa.eu/esco/skill/3556c075...,funding methods,Methoden der Mittelbeschaffung,True,False,False,optional,knowledge,cross-sector,None,NaN,NaN
89488,25214,ESCO:http://data.europa.eu/esco/skill/e36c1f47...,design ventilation network,Lüftungsnetzwerke entwerfen,True,False,False,optional,skill/competence,sector-specific,None,NaN,NaN
309630,51293,BA:K 070106-028,Automatisierungssoftware UC4,Automatisierungssoftware UC4,False,True,False,None,None,None,rule_based_1_1,2.0,2.0
515422,92434,RESUME:technical documents,RESUME:technical documents,RESUME:technical documents,False,False,True,None,None,None,method_2_1_text_mining,NaN,NaN


From the master table, you can retrieve the complete base target profile for each KldB code, as well as any extension (if available in one of the two methods).

### 6.1 Saving All Baseline Profiles + Extensions

Here, we create a summary file containing all **KldB codes as extended competency profiles**:
- df_master contains: all KLDB codes, all baseline skills, and all extensions (1.1 & 2.1)
- including flags:  src_baseline, src_ext_1_1, src_ext_2_1

In [13]:
# Export: deduplicated master table (one row per KldB code × skill)
out_master = DATA_PROCESSED_ARCHI / "kldb_profiles_baseline_plus_extension.csv"
df_master.to_csv(out_master, index=False)

print("Export OK:")
print(out_master)
print("Rows:", len(df_master))

Export OK:
C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate\kldb_profiles_baseline_plus_extension.csv
Rows: 553120


The file `kldb_profiles_baseline_plus_extension.csv` contains the consolidated competency profile
for all KldB codes. There is exactly one row for each combination of KldB code and skill.
The table combines the standardized baseline target profile (ESCO) with the data-driven
extensions from Method 1.1 (rule-based) and Method 2.1 (text mining).  
Source flags (`src_baseline`, `src_ext_1_1`, `src_ext_2_1`) ensure that the origin of each skill
remains transparently traceable.

In addition, a readable Excel file with 3 separate sheets:

In [14]:
out_xlsx = DATA_PROCESSED_ARCHI / "kldb_profiles_baseline_plus_extension_readable.xlsx"

with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    df_master[df_master["src_baseline"]].to_excel(writer, sheet_name="baseline", index=False)
    df_master[df_master["src_ext_1_1"]].to_excel(writer, sheet_name="extension_1_1", index=False)
    df_master[df_master["src_ext_2_1"]].to_excel(writer, sheet_name="extension_2_1", index=False)

print("Readable Excel OK:")
print(out_xlsx)

Readable Excel OK:
C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate\kldb_profiles_baseline_plus_extension_readable.xlsx


Readable Excel for easier review of the profiles. The CSV file contains all baseline profiles plus the extension. Here in 3 sheets:
- Baseline (ESCO/KldB)
- Extension 1.1 (rule-based)
- Extension 2.1 (text mining)

Everything is easily filterable and searchable, with no Excel row limit issues per sheet or similar problems.

Check the prefix distribution of the skill IDs, especially in Extension 2.1, to ensure that all expanded skills have been included. For Extension 1.1, the only external skill sources are ESCO and BA; for Method 2.1, LinkedIn and Resume are also included:

In [15]:
# How many source prefixes are there in total in the extension?
ext_all = df_master[(df_master["src_ext_1_1"] == True) | (df_master["src_ext_2_1"] == True)].copy()

ext_all["src_prefix"] = ext_all["skill_id"].astype(str).str.split(":", n=1).str[0]
ext_all["src_prefix"].value_counts(dropna=False)

src_prefix
BA          116304
ESCO         33798
RESUME       16472
LINKEDIN      5344
Name: count, dtype: int64

Check Columns:

In [16]:
print(df_base_min.columns)
print(df_11_min.columns)
print(df_21_min.columns)
print("master_long has skill_label_de:", "skill_label_de" in df_master_long.columns)

Index(['kldb_5_code', 'skill_id', 'skill_label', 'skill_label_de', 'src_baseline', 'src_ext_1_1', 'src_ext_2_1', 'src_name', 'relation_type', 'skill_type', 'reuse_level'], dtype='object')
Index(['kldb_5_code', 'skill_id', 'skill_label', 'skill_label_de', 'src_baseline', 'src_ext_1_1', 'src_ext_2_1', 'src_name', 'extension_method', 'ext_doc_freq', 'ext_skill_count'], dtype='object')
Index(['kldb_5_code', 'skill_id', 'skill_label', 'skill_label_de', 'src_baseline', 'src_ext_1_1', 'src_ext_2_1', 'src_name', 'extension_method'], dtype='object')
master_long has skill_label_de: True


## 7. Prioritization

In this step, a simple `priority` score is calculated for each skill-KldB pair to keep subsequent exports and models organized. In ArchiMate, hundreds of skills should not appear to be of equal importance; instead, particularly relevant candidates are sorted to the top.
- Baseline essential: higher
- Baseline optional: lower
- Extensions: additionally dependent on doc_freq

In [17]:
def compute_priority(row) -> float:
    score = 0.0
    # Baseline
    if row.get("src_baseline", False):
        score += 1.0
        # essential vs optional
        rt = str(row.get("relation_type", "")).lower()
        if "essential" in rt:
            score += 1.0  # essential boost
        elif "optional" in rt:
            score += 0.2

    # Expansions
    if row.get("src_ext_1_1", False):
        score += 0.6
    if row.get("src_ext_2_1", False):
        score += 0.6 # 1.1 & 2.1 gleich gewichtet

    # doc_freq
    dfreq = row.get("doc_freq", None)
    if dfreq is None:
        dfreq = row.get("ext_doc_freq", None)
    if dfreq is not None and not pd.isna(dfreq):
        # log dampening
        score += float(np.log1p(dfreq)) * 0.25

    return float(score)

df_master["priority"] = df_master.apply(compute_priority, axis=1)
print(df_master["priority"].describe())
display(df_master.sort_values("priority", ascending=False).head(20))

count    553120.000000
mean          1.407715
std           0.442603
min           0.600000
25%           1.200000
50%           1.200000
75%           2.000000
max           3.224661
Name: priority, dtype: float64


,kldb_5_code,skill_id,skill_label,skill_label_de,src_baseline,src_ext_1_1,src_ext_2_1,relation_type,skill_type,reuse_level,extension_method,ext_doc_freq,ext_skill_count,priority
119373,27103,BA:K 0705-170,Spring Framework,Spring Framework,False,True,True,None,None,None,rule_based_1_1,3289.0,3289.0,3.224661
118880,27103,BA:K 0700-070,Microsoft 365,Microsoft 365,False,True,True,None,None,None,rule_based_1_1,2404.0,2404.0,3.146326
119408,27103,BA:K 0705-218,Django Framework,Django Framework,False,True,True,None,None,None,rule_based_1_1,2401.0,2401.0,3.146014
405904,72144,ESCO:http://data.europa.eu/esco/skill/7b5cce4d...,computer science,Informatik,False,True,True,None,None,None,rule_based_1_1,1328.0,1328.0,2.998046
404687,72144,BA:K 0700-062,Security Automation,Security Automation,False,True,True,None,None,None,rule_based_1_1,1213.0,1213.0,2.975419
291598,43353,ESCO:http://data.europa.eu/esco/skill/7b5cce4d...,computer science,Informatik,False,True,True,None,None,None,rule_based_1_1,989.0,989.0,2.924426
287545,43343,BA:K 0700-070,Microsoft 365,Microsoft 365,False,True,True,None,None,None,rule_based_1_1,911.0,911.0,2.903910
119295,27103,BA:K 0705-052,Programmiersprache JavaScript,Programmiersprache JavaScript,False,True,False,None,None,None,rule_based_1_1,8276.0,8276.0,2.855309
187157,31114,BA:K 0700-070,Microsoft 365,Microsoft 365,False,True,True,None,None,None,rule_based_1_1,739.0,739.0,2.851663
119293,27103,BA:K 0705-050,Programmiersprache Java,Programmiersprache Java,False,True,False,None,None,None,rule_based_1_1,7689.0,7689.0,2.836919


Prioritization is not an evaluation, but rather a pragmatic mechanism designed to generate a small, explainable selection for the demo from very large sets of skills. Weighting: Baseline-Essential > Baseline-Optional; extensions are additionally dampened via the frequency signal (doc_freq).

Important columns in the output:
- `ext_doc_freq`: Number of documents (e.g., external profiles/job descriptions) in which a skill was found as part of the profile extension. Found more frequently = tends to be a more robust/typical skill candidate.
- `ext_skill_count`: Frequency of the skill (total number of mentions). Higher = the skill appears more often, but this may also favor generic terms.
- `priority`: Combined score based on: Baseline (essential weighted more heavily than optional), profiler extension (1.1/2.1); skills from extensions receive a bonus.
- `ext_doc_freq` is included in the calculation with a damping factor (so that very frequent skills do not dominate everything)

## 8. Selection of Specific Actual Roles (KldB Codes)

Define KldB codes as actual inputs. The 4 codes from the actual model, entered manually:
- Max Meier (Field Representative): 61123
- Luise Schneider (Technical Sales Assistant): 61122
- Manuel Müller (Mechanical Engineering Operations Engineer): 27304
- Tim Lange (Plant Installer/Assembly Mechanic): 34342

The KldB codes for the four competency profiles were manually extracted.

In [18]:
# Mapping structure
IST_ACTORS = [
    {"actor_id": "IST_MAX", "actor_name": "Max Maier", "role_label": "Außendienstmitarbeiter/in", "kldb_5_code": "61123"},
    {"actor_id": "IST_LUISE", "actor_name": "Luise Schneider", "role_label": "Assistent/in (technisch) – Vertrieb", "kldb_5_code": "61122"},
    {"actor_id": "IST_MANUEL", "actor_name": "Manuel Müller", "role_label": "Betriebsingenieur/in (Maschinenbau)", "kldb_5_code": "27304"},
    {"actor_id": "IST_TIM", "actor_name": "Tim Lange", "role_label": "Anlagenmonteur/in", "kldb_5_code": "34342"},
]

## 9. Profile Logic: Separating Baseline from Extension

Baseline/Base Profiles (KldB → ESCO) map the standardized target profile from the official taxonomy:
- Essential: All are included in the export; this reflects the standard occupational reality
- Optional: included only to a limited extent as supplements
- Hard limits or subsequent manual selection to prevent an excessive number of skills or an overly large model.

Extensions from previous steps in the process provide additional skill candidates from external data sources:
- Extensions may contain noise (e.g., generic terms, matching errors, skills outside the domain, terms that do not represent skills) and are therefore intentionally exported as a candidate pool.
- The top 10–15 skills should be presented in the model according to priority. A larger number of skills is exported here, since a manual selection will be performed afterward anyway. This way, we obtain a clear, uncluttered demo model.
- The system automatically generates candidates -> a human validates/curates them for modeling.
- Additionally, it makes sense to display the source (1.1/2.1) for each skill in the export.

The baseline is official/standardized; the extension is customized/derived empirically. This ensures transparency regarding which skills originate from the taxonomy and which are added through profile extension.

In [19]:
# Help Function: Build Profile for KldB Codes (Baseline + Extension Top-N)
def build_profile_for_kldb(df_master: pd.DataFrame, kldb_code: str, top_ext: int = 70, top_optional_baseline: int = 20) -> dict:

    kldb_code = norm_kldb_code(kldb_code)
    df_k = df_master[df_master["kldb_5_code"] == kldb_code].copy()
    # Safety
    if df_k.empty:
        return {"kldb_5_code": kldb_code, "baseline": pd.DataFrame(), "extension": pd.DataFrame()}

    # Baseline split essential/optional, relation_type from ESCO
    df_base = df_k[df_k["src_baseline"] == True].copy()
    df_base["relation_type"] = df_base.get("relation_type", "").fillna("").astype(str)

    df_base_essential = df_base[df_base["relation_type"].str.lower().str.contains("essential")].copy()
    df_base_optional  = df_base[df_base["relation_type"].str.lower().str.contains("optional")].copy()

    # limit optional
    if len(df_base_optional) > top_optional_baseline:
        if "reuse_level" in df_base_optional.columns:
            df_base_optional = df_base_optional.sort_values(["reuse_level", "skill_label"]).head(top_optional_baseline)
        else:
            df_base_optional = df_base_optional.sort_values(["skill_label"]).head(top_optional_baseline)

    df_baseline_final = pd.concat([df_base_essential, df_base_optional], ignore_index=True)

    # Extension from non-baseline + 1.1/2.1
    # df_ext = df_k[(df_k["src_ext_1_1"] == True) | (df_k["src_ext_2_1"] == True)].copy()
    df_ext = df_k[(~df_k["src_baseline"]) & (df_k["src_ext_1_1"] | df_k["src_ext_2_1"])].copy()

    # sort by priority
    if "priority" in df_ext.columns:
        df_ext = df_ext.sort_values(["priority", "skill_label"], ascending=[False, True])
    else:
        df_ext = df_ext.sort_values(["skill_label"])

    # Enforce the minimum percentage of skills from 2.1, since 1.1 covers a broader range; otherwise, only skills from 1.1 would be included
    min_21 = 10  # 10 Skills per KldB from 2.1
    df_ext_21 = df_ext[df_ext["src_ext_2_1"] == True].head(min_21)
    df_ext_rest = df_ext.drop(df_ext_21.index).head(max(0, top_ext - len(df_ext_21)))

    df_ext_final = pd.concat([df_ext_21, df_ext_rest], ignore_index=True)

    df_ext_final["extension_method"] = df_ext_final.apply( # specified method
        lambda r: "text_mining_2_1" if r.get("src_ext_2_1", False) else ("rule_based_1_1" if r.get("src_ext_1_1", False) else None),
        axis=1
    )

    return {"kldb_5_code": kldb_code,"baseline": df_baseline_final,"extension": df_ext_final}

In [20]:
# Create profiles for all current employees
profiles = []
for a in IST_ACTORS:
    prof = build_profile_for_kldb(df_master, a["kldb_5_code"], top_ext=70, top_optional_baseline=20)
    profiles.append({**a, **prof})

# Output Results
for p in profiles:
    print(p["actor_name"], p["kldb_5_code"], "baseline:", len(p["baseline"]), "extension:", len(p["extension"]))

Max Maier 61123 baseline: 65 extension: 70
Luise Schneider 61122 baseline: 39 extension: 70
Manuel Müller 27304 baseline: 183 extension: 70
Tim Lange 34342 baseline: 174 extension: 70


## 10. Generate ArchiMate Input Tables

2 Export Types:
- Elements: Which skills should exist as elements? A list of all skills for the AS-IS model
- Relationships: Which actors have which skills? A list of skill links for each person, including a flag for baseline vs. extension

Output Documents:
- archi_ist_skill_elements.csv
- archi_ist_actor_skill_links.csv
- archi_ist_profiles_readable.xlsx (as an overview for manual review)

In [21]:
# Skill element list, unnique
rows_skills = []
rows_links = []

for p in profiles:
    actor_id = p["actor_id"]
    actor_name = p["actor_name"]

    for group_name, df_part in [("baseline", p["baseline"]), ("extension", p["extension"])]:
        if df_part is None or df_part.empty:
            continue

        df_part = df_part.copy()
        df_part["group"] = group_name
        df_part["actor_id"] = actor_id
        df_part["actor_name"] = actor_name

        # Link Table
        keep_link_cols = ["actor_id","actor_name","kldb_5_code","skill_id","skill_label","skill_label_de", "group","priority","src_baseline","src_ext_1_1","src_ext_2_1","relation_type"]
        for c in keep_link_cols:
            if c not in df_part.columns:
                df_part[c] = None

        rows_links.append(df_part[keep_link_cols])

        # Skill Elements
        keep_skill_cols = ["skill_id","skill_label","skill_label_de","src_baseline","src_ext_1_1","src_ext_2_1"]
        for c in keep_skill_cols:
            if c not in df_part.columns:
                df_part[c] = None
        rows_skills.append(df_part[keep_skill_cols])

df_archi_links = pd.concat(rows_links, ignore_index=True).drop_duplicates()
df_archi_skills = (
    pd.concat(rows_skills, ignore_index=True)
    .drop_duplicates(subset=["skill_id"])
    .reset_index(drop=True)
)

print("Links:", df_archi_links.shape, "Skills:", df_archi_skills.shape)
display(df_archi_links.head(5))
display(df_archi_skills.head(5))

Links: (741, 12) Skills: (586, 6)


,actor_id,actor_name,kldb_5_code,skill_id,skill_label,skill_label_de,group,priority,src_baseline,src_ext_1_1,src_ext_2_1,relation_type
0,IST_MAX,Max Maier,61123,ESCO:http://data.europa.eu/esco/skill/00e53a0a...,guarantee customer satisfaction,Kundenzufriedenheit gewährleisten,baseline,2.0,True,False,False,essential
1,IST_MAX,Max Maier,61123,ESCO:http://data.europa.eu/esco/skill/05e5e14f...,solar energy,Sonnenenergie,baseline,2.0,True,False,False,essential
2,IST_MAX,Max Maier,61123,ESCO:http://data.europa.eu/esco/skill/0c0488b3...,sales argumentation,Verkaufsargumentation,baseline,2.0,True,False,False,essential
3,IST_MAX,Max Maier,61123,ESCO:http://data.europa.eu/esco/skill/0da516ee...,communicate with customers,mit Kunden kommunizieren,baseline,2.0,True,False,False,essential
4,IST_MAX,Max Maier,61123,ESCO:http://data.europa.eu/esco/skill/0dd4d328...,consumer goods industry,Konsumgüterindustrie,baseline,2.0,True,False,False,essential


,skill_id,skill_label,skill_label_de,src_baseline,src_ext_1_1,src_ext_2_1
0,ESCO:http://data.europa.eu/esco/skill/00e53a0a...,guarantee customer satisfaction,Kundenzufriedenheit gewährleisten,True,False,False
1,ESCO:http://data.europa.eu/esco/skill/05e5e14f...,solar energy,Sonnenenergie,True,False,False
2,ESCO:http://data.europa.eu/esco/skill/0c0488b3...,sales argumentation,Verkaufsargumentation,True,False,False
3,ESCO:http://data.europa.eu/esco/skill/0da516ee...,communicate with customers,mit Kunden kommunizieren,True,False,False
4,ESCO:http://data.europa.eu/esco/skill/0dd4d328...,consumer goods industry,Konsumgüterindustrie,True,False,False


## 11. Export for Archi Modeling

Export the files to the processed_archimate folder:

In [22]:
out_skills = DATA_PROCESSED_ARCHI / "archi_ist_skill_elements.csv"
out_links  = DATA_PROCESSED_ARCHI / "archi_ist_actor_skill_links.csv"

df_archi_skills.to_csv(out_skills, index=False)
df_archi_links.to_csv(out_links, index=False)

print("Export OK:")
print(out_skills)
print(out_links)

Export OK:
C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate\archi_ist_skill_elements.csv
C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate\archi_ist_actor_skill_links.csv


- `archi_ist_skill_elements.csv` contains all skills used in the current (IST) model as unique skill elements. For each skill, the file includes the ID, description (DE/EN), and source (Baseline, Profile Extension 1.1/2.1)
- `archi_ist_actor_skill_links.csv` describes the assignment of skills to current (IST) actors. Each row represents a relationship between an employee and a skill, including group membership (Baseline/Extension), priority, and origin. This table forms the basis for the skill assignments in the ArchiMate model.

**Summary Excel Overview by Employee:**
 
 `archi_ist_profiles_readable.xlsx` clearly presents employees' current competency profiles. For each employee, there is one sheet with basic profile skills and one sheet with skills from the advanced methods.

In [23]:
out_xlsx = DATA_PROCESSED_ARCHI / "archi_ist_profiles_readable.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
    for p in profiles:
        name = p["actor_name"].replace("/", "_")[:31]
        p["baseline"].to_excel(writer, sheet_name=f"{name}_baseline", index=False)
        p["extension"].to_excel(writer, sheet_name=f"{name}_ext", index=False)

print("Readable XLSX:", out_xlsx)

Readable XLSX: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_archimate\archi_ist_profiles_readable.xlsx


# Conclusion: Notebook 12

In this notebook, all relevant skill data for the current (IST) model was consolidated, standardized, and prepared in such a way that it can be used directly for modeling in ArchiMate. The starting point was the standardized baseline skills from ESCO, supplemented by skills derived from the rule-based (Method 1.1) and text-based profile extension (Method 2.1).

Key steps and results:
- Consolidation of all skill sources into a single database
- Standardization of KldB codes, skill IDs, and skill labels
- Linguistic enrichment through German skill labels (remaining English skills may be manually translated in the current model if necessary)
- Introduction of a priority score to better classify the relevance and importance of skills (including baseline status, extension source, frequency)

Based on this, role-based current-state profiles were created that clearly distinguish between baseline skills and extended skills. These profiles form a transparent and traceable basis for the manual ArchiMate modeling of the current state. In the next step, the current-state model will be manually developed in ArchiMate. Building on this, Notebook 13 is used to perform a structured matching process with the target model to identify specific use cases.

**Note on Profile Extension Result Files:**

In addition, two consolidated result files were generated (Step 6.1), which serve as the central output of the profile extension. The folder `data/processed_archimate/` contains the file `kldb_profiles_baseline_plus_extension.csv`, which includes the complete baseline target profile (ESCO) for all KldB-5 roles, including all data-driven extensions from Methods 1.1 and 2.1. The file is structured on a line-by-line basis (one line per KldB code × skill) and contains, among other things, skill IDs, German and English skill labels, and source flags (src_baseline, src_ext_1_1, src_ext_2_1).

In addition, a readable Excel overview was generated in the same folder (`archi_ist_profiles_readable.xlsx`), which presents the extended current profiles in a structured manner.
Together, both files form a complete, transparent, and reproducible representation of the extended competency profiles as the result of the work.

# Modeling the Extended current (IST) Model

In the ArchiMate folder (Kompetenzabgleich_neu_extended.archimate and Kompetenzabgleich_neu_extended.xml). Further development of the provided ArchiMate model.

**Modeling Decisions**
- Skills as Resources (not Capabilities): Individual competencies are modeled as resource elements, since capabilities describe organizational abilities, while skills represent individual resources (Azevedo et al. 2015; Calhau et al. 2024).
- Separation of Baseline vs. Extension: For each employee, standardized KldB/ESCO skills (baseline) and data-driven extended skills (profile extension) are modeled with a clear visual distinction to indicate their origin and reliability (Calhau et al. 2024; Calhau & Almeida 2023; Martić et al. 2024).

**Data Basis**
- The basis is the “Readable Excel” file generated here in Notebook 12, with separate sheets for Baseline (`*_baseline`) and Extension (`*_ext`) per employee (data\processed_archimate\archi_ist_profiles_readable).
- The Excel file was manually reviewed and cleaned before the skills were imported into ArchiMate.

**Manual Validation & Selection of Extended Skills**
- The automatically generated file `archi_ist_profiles_readable` was copied and saved as `archi_ist_profiles_readable_manual_selection` in the `data/processed_archimate` folder to serve as a working basis.
- For each employee, the worksheet containing the extended profiles (*_ext) was copied (labeled _MANUAL) and manually reviewed.
- Obviously incorrect entries, technically inappropriate terms, duplicates, and entries that did not clearly represent a skill were removed. The skills actually incorporated into ArchiMate are highlighted in green in the table to ensure traceability between the database and the model.
- The baseline profiles (*_baseline) were not modified but used exclusively as a reference. During the manual selection process, care was taken not to include redundant or very similar skills that were already contained in the baseline target profile.
- The goal of this step is to qualitatively validate the data-driven profile expansion and to select additional competencies that are professionally plausible and relevant to modeling.

**Selection & Reduction**
- Baseline: 15 core, predominantly essential skills per person.
- Expansion: approx. 10–15 skills per person, after removing mismatches, duplicates, and existing baseline skills.
- English skill labels were manually translated into German as needed.
- The reduction serves to improve the readability and demo suitability of the ArchiMate view.

**ArchiMate Modeling**
- Two grouping elements per employee:
- Baseline Skills
- Extended Skills
- Relationships:
- Person -> Grouping: Association
- Grouping -> Skill (Resource): Composition
- The approach corresponds to that of the original reference model.

**Observations**
- Baseline skills are highly standardized and, in some cases, general.
- Extended skills are more heterogeneous and include both specific tools/methods (e.g., for Manuel Müller, an industrial engineer (mechanical engineering), the skills: project planning software MS Project (MS Office) or project management according to PRINCE2) as well as general, cross-disciplinary competencies (e.g., communication, teamwork, working independently). Since the advanced skills are, for example, “more flexible” than standardized skills, they appear broader overall and include fewer formal terms—such as an increased emphasis on soft or foundational skills.
- This distinction makes these differences explicitly visible and forms the basis for expanded target profiles.